In [1]:
# ============================================================
# CS 499 Milestone Two
# Enhancement One: Software Design and Engineering
# Animal Shelter Dashboard
# Samari Robinson Camacho
# ============================================================

# Dash imports
from dash import Dash, html, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl

# Data-processing import
import pandas as pd

# Import the enhanced CRUD module
from CRUD_Python_Module import CRUD


# ============================================================
# Application Configuration
# ============================================================

APP_TITLE = "SNHU CS-340 Animal Shelter Dashboard"
AUTHOR_NAME = "Samari Robinson Camacho"

DATABASE_HOST = "localhost"
DATABASE_PORT = 27017
DATABASE_NAME = "aac"
COLLECTION_NAME = "animals"

DEFAULT_MAP_CENTER = [30.75, -97.48]
DEFAULT_MAP_ZOOM = 10

TABLE_PAGE_SIZE = 10
MAP_WIDTH = "100%"
MAP_HEIGHT = "500px"


# ============================================================
# Database Connection
# ============================================================

def create_database_connection():
    """
    Connect to the local MongoDB Community Server.

    The new development computer currently uses a local MongoDB
    installation without username and password authentication.
    """

    try:
        return CRUD(
            host=DATABASE_HOST,
            port=DATABASE_PORT,
            database_name=DATABASE_NAME,
            collection_name=COLLECTION_NAME
        )

    except Exception as error:
        raise ConnectionError(
            f"Unable to connect to MongoDB: {error}"
        ) from error


# ============================================================
# Data Retrieval and Preparation
# ============================================================

def load_animal_records(database):
    """
    Retrieve animal records from MongoDB and prepare them for Dash.

    MongoDB ObjectId values are removed because they cannot be
    displayed directly by the Dash DataTable.
    """

    try:
        records = database.read({})

        if records is None:
            records = []

        if not isinstance(records, list):
            records = list(records)

        dataframe = pd.DataFrame.from_records(records)

        if "_id" in dataframe.columns:
            dataframe.drop(
                columns=["_id"],
                inplace=True
            )

        dataframe.fillna("", inplace=True)

        return dataframe

    except Exception as error:
        print(f"Unable to retrieve animal records: {error}")
        return pd.DataFrame()


def find_column(dataframe, possible_names):
    """
    Find a dataset column using a list of possible names.

    This replaces fragile numerical column positions with meaningful
    column names so the program remains maintainable if column order
    changes.
    """

    normalized_columns = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for possible_name in possible_names:
        normalized_name = possible_name.strip().lower()

        if normalized_name in normalized_columns:
            return normalized_columns[normalized_name]

    return None


def create_table_columns(dataframe):
    """
    Create the Dash DataTable column configuration.
    """

    return [
        {
            "name": column,
            "id": column,
            "deletable": False,
            "selectable": True
        }
        for column in dataframe.columns
    ]


# ============================================================
# Initialize Database and Data
# ============================================================

try:
    shelter = create_database_connection()
    df = load_animal_records(shelter)

except Exception as error:
    print(error)
    shelter = None
    df = pd.DataFrame()


# Identify important fields by name instead of fixed positions
latitude_column = find_column(
    df,
    [
        "location_lat",
        "latitude",
        "lat"
    ]
)

longitude_column = find_column(
    df,
    [
        "location_long",
        "location_lon",
        "longitude",
        "long",
        "lon"
    ]
)

breed_column = find_column(
    df,
    [
        "breed"
    ]
)

name_column = find_column(
    df,
    [
        "name",
        "animal_name"
    ]
)


# ============================================================
# Dashboard Layout / View
# ============================================================

app = Dash(__name__)

app.title = APP_TITLE

app.layout = html.Div(
    children=[
        html.Div(
            children=[
                html.H1(
                    APP_TITLE,
                    style={
                        "textAlign": "center",
                        "marginBottom": "5px"
                    }
                ),

                html.H3(
                    f"Created by {AUTHOR_NAME}",
                    style={
                        "textAlign": "center",
                        "fontWeight": "normal",
                        "marginTop": "0"
                    }
                )
            ]
        ),

        html.Hr(),

        html.H2("Animal Shelter Records"),

        html.P(
            "Use the table controls to sort or filter the records. "
            "Select one row to display the animal's location on the map."
        ),

        dash_table.DataTable(
            id="datatable-id",

            columns=create_table_columns(df),

            data=df.to_dict("records"),

            row_selectable="single",

            selected_rows=[0] if not df.empty else [],

            page_size=TABLE_PAGE_SIZE,

            sort_action="native",

            filter_action="native",

            style_table={
                "overflowX": "auto"
            },

            style_header={
                "fontWeight": "bold",
                "textAlign": "left",
                "backgroundColor": "#E8E8E8"
            },

            style_cell={
                "textAlign": "left",
                "minWidth": "120px",
                "width": "120px",
                "maxWidth": "180px",
                "whiteSpace": "normal",
                "height": "auto",
                "padding": "8px"
            }
        ),

        html.Br(),

        html.Hr(),

        html.H2("Selected Animal Location"),

        html.Div(
            id="map-message-id"
        ),

        html.Div(
            id="map-id",
            className="col s12 m6"
        )
    ],

    style={
        "padding": "20px",
        "fontFamily": "Arial, sans-serif"
    }
)


# ============================================================
# Dashboard Controller
# ============================================================

@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),
    Input(
        "datatable-id",
        "selected_columns"
    )
)
def update_styles(selected_columns):
    """
    Highlight columns selected by the dashboard user.
    """

    if not selected_columns:
        return []

    return [
        {
            "if": {
                "column_id": column
            },
            "backgroundColor": "#D2F3FF",
            "fontWeight": "bold"
        }
        for column in selected_columns
    ]


@app.callback(
    [
        Output(
            "map-id",
            "children"
        ),

        Output(
            "map-message-id",
            "children"
        )
    ],
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),

        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(view_data, selected_rows):
    """
    Update the map using the selected animal record.

    Defensive validation prevents crashes when data, coordinates,
    columns, or row selections are missing or invalid.
    """

    if not view_data:
        return [], html.P(
            "No animal records are currently available."
        )

    filtered_dataframe = pd.DataFrame.from_records(
        view_data
    )

    if filtered_dataframe.empty:
        return [], html.P(
            "No records match the current table filters."
        )

    if latitude_column is None or longitude_column is None:
        return [], html.P(
            "The dataset does not contain recognized "
            "latitude and longitude columns."
        )

    if selected_rows:
        row_index = selected_rows[0]
    else:
        row_index = 0

    if row_index < 0 or row_index >= len(filtered_dataframe):
        row_index = 0

    selected_record = filtered_dataframe.iloc[row_index]

    try:
        latitude = float(
            selected_record[latitude_column]
        )

        longitude = float(
            selected_record[longitude_column]
        )

    except (TypeError, ValueError, KeyError):
        return [], html.P(
            "The selected record does not contain valid coordinates."
        )

    animal_name = "Name unavailable"
    animal_breed = "Breed unavailable"

    if name_column and name_column in selected_record.index:
        name_value = str(
            selected_record[name_column]
        ).strip()

        if name_value:
            animal_name = name_value

    if breed_column and breed_column in selected_record.index:
        breed_value = str(
            selected_record[breed_column]
        ).strip()

        if breed_value:
            animal_breed = breed_value

    animal_map = dl.Map(
        style={
            "width": MAP_WIDTH,
            "height": MAP_HEIGHT
        },

        center=[
            latitude,
            longitude
        ],

        zoom=DEFAULT_MAP_ZOOM,

        children=[
            dl.TileLayer(
                id="base-layer-id"
            ),

            dl.Marker(
                position=[
                    latitude,
                    longitude
                ],

                children=[
                    dl.Tooltip(
                        animal_breed
                    ),

                    dl.Popup(
                        children=[
                            html.H3(
                                "Animal Information"
                            ),

                            html.P(
                                f"Name: {animal_name}"
                            ),

                            html.P(
                                f"Breed: {animal_breed}"
                            ),

                            html.P(
                                f"Coordinates: "
                                f"{latitude:.5f}, "
                                f"{longitude:.5f}"
                            )
                        ]
                    )
                ]
            )
        ]
    )

    map_message = html.P(
        f"Displaying the shelter location for {animal_name}.",
        style={
            "fontWeight": "bold"
        }
    )

    return animal_map, map_message


# ============================================================
# Run the Dashboard
# ============================================================

if df.empty:
    print(
        "MongoDB is connected, but no animal records were loaded. "
        "Import the animal shelter dataset into the 'aac' database "
        "and the 'animals' collection."
    )

# Change the port to 8051 if port 8050 is already being used.
app.run(
    port=8050,
    debug=False
)

Connected to MongoDB database 'aac' and collection 'animals'.
